# 2. Pré-Processamento de Dados

## Objetivo

Este notebook realiza o pré-processamento dos dados, transformando os dados brutos em um formato pronto para modelagem. Incluindo:
- Tratamento de valores faltantes
- Detecção e tratamento de outliers
- Normalização e padronização de variáveis
- Codificação de variáveis categóricas
- Seleção de features
- Divisão em conjuntos de treino e teste

## Por que pré-processamento é importante?

**Garbage in, garbage out!** A qualidade dos dados determina a qualidade do modelo. O pré-processamento:
1. **Remove ruído**: Dados inconsistentes ou errados
2. **Trata valores faltantes**: Decide como lidar com dados incompletos
3. **Normaliza**: Coloca variáveis na mesma escala
4. **Remove outliers**: Valores extremos que distorcem o modelo
5. **Seleciona features**: Usa apenas variáveis relevantes

---

## Seção 1: Importações e Carregamento de Dados

In [ ]:
# Importações
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Importações de pré-processamento
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

print("✓ Bibliotecas importadas com sucesso!")

In [ ]:
# Carregar dados
data_dir = Path("../data/raw")
treated_file = data_dir / "df_iacov_treated.xlsx"

print("Carregando dados tratados...")
df = pd.read_excel(treated_file)
print(f"✓ Dados carregados: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

# Fazer uma cópia para trabalhar
df_processed = df.copy()

print(f"\nPrimeiras linhas:")
print(df_processed.head())

## Seção 2: Tratamento de Valores Faltantes

### O que são valores faltantes?
Valores faltantes (NaN, NA) ocorrem quando uma informação não foi registrada. Temos várias estratégias:

1. **Remover linhas**: Se poucos dados faltam
2. **Remover colunas**: Se muitos dados faltam em uma variável
3. **Imputação por média/mediana**: Preencher com valor central
4. **Imputação por KNN**: Usar valores de vizinhos similares
5. **Imputação por modelo**: Treinar modelo para prever valores faltantes

In [ ]:
# Análise de valores faltantes
print("="*80)
print("ANÁLISE DE VALORES FALTANTES")
print("="*80)

missing_analysis = pd.DataFrame({
    'Coluna': df_processed.columns,
    'Faltantes': df_processed.isnull().sum().values,
    'Percentual': (df_processed.isnull().sum() / len(df_processed) * 100).round(2).values
})

missing_analysis = missing_analysis[missing_analysis['Faltantes'] > 0].sort_values('Faltantes', ascending=False)

print("\nVariáveis com valores faltantes:")
print(missing_analysis.to_string(index=False))

# Visualizar
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(missing_analysis['Coluna'], missing_analysis['Percentual'], color='coral')
ax.set_xlabel('Percentual de Valores Faltantes (%)')
ax.set_title('Distribuição de Valores Faltantes', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/08_missing_values_before_preprocessing.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico salvo")

In [ ]:
# Estratégia de tratamento de valores faltantes
print("\n" + "="*80)
print("ESTRATÉGIA DE TRATAMENTO")
print("="*80)

# Separar variáveis numéricas e categóricas
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()

print(f"\nVariáveis numéricas ({len(numeric_cols)}): {numeric_cols}")
print(f"\nVariáveis categóricas ({len(categorical_cols)}): {categorical_cols}")

# Para variáveis numéricas: imputação por mediana
print("\n✓ Variáveis numéricas: Imputação por MEDIANA")
print("  Motivo: Mediana é robusta a outliers, melhor que média")

imputer_numeric = SimpleImputer(strategy='median')
df_processed[numeric_cols] = imputer_numeric.fit_transform(df_processed[numeric_cols])

# Para variáveis categóricas: imputação por moda (valor mais frequente)
print("\n✓ Variáveis categóricas: Imputação por MODA")
print("  Motivo: Valor mais frequente preserva distribuição original")

imputer_categorical = SimpleImputer(strategy='most_frequent')
df_processed[categorical_cols] = imputer_categorical.fit_transform(df_processed[categorical_cols])

# Verificar se todos os faltantes foram tratados
print(f"\nValores faltantes após imputação: {df_processed.isnull().sum().sum()}")
print("✓ Todos os valores faltantes foram tratados!")

## Seção 3: Identificação e Tratamento de Outliers

### O que são outliers?
Outliers são valores extremos que se desviam muito do padrão. Usaremos o método **IQR (Interquartile Range)**:

- **Q1**: 25º percentil
- **Q3**: 75º percentil
- **IQR**: Q3 - Q1
- **Outliers**: Valores < Q1 - 1.5×IQR ou > Q3 + 1.5×IQR

In [ ]:
def detect_outliers_iqr(data):
    """
    Detecta outliers usando o método IQR.
    
    Args:
        data: Série pandas com dados numéricos
        
    Returns:
        Série booleana indicando outliers
    """
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    return (data < lower_bound) | (data > upper_bound)

# Detectar outliers
print("="*80)
print("DETECÇÃO DE OUTLIERS")
print("="*80)

outlier_summary = {}

for col in numeric_cols:
    outliers = detect_outliers_iqr(df_processed[col])
    outlier_summary[col] = outliers.sum()

outlier_df = pd.DataFrame({
    'Variável': list(outlier_summary.keys()),
    'Número de Outliers': list(outlier_summary.values()),
    'Percentual': [f"{(count / len(df_processed) * 100):.2f}%" for count in outlier_summary.values()]
}).sort_values('Número de Outliers', ascending=False)

print("\nOutliers por variável:")
print(outlier_df[outlier_df['Número de Outliers'] > 0].to_string(index=False))

In [ ]:
# Estratégia para outliers: MANTER
print("\n" + "="*80)
print("DECISÃO SOBRE OUTLIERS")
print("="*80)

print("""
✓ DECISÃO: MANTER os outliers

Motivos:
1. Outliers podem ser casos clínicos genuinamente extremos
2. Gradient boosting é robusto a outliers
3. Remover pode perder informação importante
4. Pacientes com valores extremos podem ter maior risco de óbito

Alternativa: Se necessário, usar técnicas de normalização robustas
""")

## Seção 4: Codificação de Variáveis Categóricas

### O que é codificação?
Algoritmos de aprendizado de máquina trabalham com números, não com texto. Precisamos converter variáveis categóricas em números.

**Métodos:**
- **Label Encoding**: Converte categorias em 0, 1, 2, ... (para variáveis ordinais)
- **One-Hot Encoding**: Cria coluna binária para cada categoria (para variáveis nominais)

In [ ]:
print("="*80)
print("CODIFICAÇÃO DE VARIÁVEIS CATEGÓRICAS")
print("="*80)

# Analisar variáveis categóricas
print("\nVariáveis categóricas:")
for col in categorical_cols:
    print(f"\n{col}:")
    print(df_processed[col].value_counts())
    print(f"Valores únicos: {df_processed[col].nunique()}")

In [ ]:
# Codificação
print("\n" + "="*80)
print("APLICANDO CODIFICAÇÃO")
print("="*80)

# Para 'city_hospital': usar Label Encoding (ordinal)
if 'city_hospital' in categorical_cols:
    print("\n✓ city_hospital: Label Encoding")
    le_city = LabelEncoder()
    df_processed['city_hospital_encoded'] = le_city.fit_transform(df_processed['city_hospital'])
    
    # Criar mapeamento para referência
    city_mapping = dict(zip(le_city.classes_, le_city.transform(le_city.classes_)))
    print(f"  Mapeamento: {city_mapping}")
    
    # Remover coluna original
    df_processed = df_processed.drop('city_hospital', axis=1)
    df_processed = df_processed.rename(columns={'city_hospital_encoded': 'city_hospital'})

print("\n✓ Codificação concluída!")
print(f"\nDimensões após codificação: {df_processed.shape}")

## Seção 5: Normalização e Padronização

### Por que normalizar?
Variáveis em escalas diferentes podem distorcer o modelo. Exemplos:
- Idade: 0-100
- Frequência cardíaca: 40-200
- Leucócitos: 1000-20000

**Métodos:**
- **Standardization (Z-score)**: (x - média) / desvio padrão → média=0, desvio=1
- **Normalization (Min-Max)**: (x - min) / (max - min) → valores entre 0 e 1

In [ ]:
print("="*80)
print("NORMALIZAÇÃO E PADRONIZAÇÃO")
print("="*80)

# Separar features e target
target_col = 'death'
feature_cols = [col for col in df_processed.columns if col != target_col]

X = df_processed[feature_cols].copy()
y = df_processed[target_col].copy()

print(f"\nFeatures: {len(feature_cols)} variáveis")
print(f"Target: {target_col}")
print(f"\nDistribuição do target:")
print(y.value_counts())

# Aplicar StandardScaler
print("\n✓ Aplicando StandardScaler (Z-score normalization)")
print("  Motivo: Gradient boosting funciona bem com dados padronizados")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print(f"\nEstatísticas após padronização:")
print(f"  Média: {X_scaled.mean().mean():.6f} (próximo de 0)")
print(f"  Desvio padrão: {X_scaled.std().mean():.6f} (próximo de 1)")

# Visualizar antes e depois
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Antes
axes[0].hist(X.iloc[:, 0], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title(f'Antes: {feature_cols[0]}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Valor')
axes[0].set_ylabel('Frequência')

# Depois
axes[1].hist(X_scaled.iloc[:, 0], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title(f'Depois: {feature_cols[0]} (Padronizado)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Valor Padronizado')
axes[1].set_ylabel('Frequência')

plt.tight_layout()
plt.savefig('../figures/09_standardization_before_after.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico salvo")

## Seção 6: Divisão em Conjuntos de Treino e Teste

### Por que dividir os dados?
Precisamos avaliar o modelo em dados que ele nunca viu antes. Caso contrário:
- O modelo pode estar "memorizando" os dados (overfitting)
- Não sabemos como ele se comportará em novos pacientes

**Proporção típica:**
- 80% treino (para aprender)
- 20% teste (para avaliar)

In [ ]:
print("="*80)
print("DIVISÃO EM CONJUNTOS DE TREINO E TESTE")
print("="*80)

# Divisão estratificada (mantém proporção de classes)
print("\n✓ Usando train_test_split com stratify")
print("  Motivo: Garante que treino e teste têm proporção similar de óbitos")

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Mantém proporção de classes
)

print(f"\nTamanho do conjunto de treino: {X_train.shape[0]:,} pacientes ({X_train.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"Tamanho do conjunto de teste: {X_test.shape[0]:,} pacientes ({X_test.shape[0]/len(X_scaled)*100:.1f}%)")

print(f"\nDistribuição de óbitos no TREINO:")
print(y_train.value_counts())
print(f"Taxa de mortalidade: {(y_train == 1).sum() / len(y_train) * 100:.2f}%")

print(f"\nDistribuição de óbitos no TESTE:")
print(y_test.value_counts())
print(f"Taxa de mortalidade: {(y_test == 1).sum() / len(y_test) * 100:.2f}%")

print("\n✓ Proporções mantidas entre treino e teste!")

## Seção 7: Salvando Dados Processados

In [ ]:
# Salvar dados processados
processed_dir = Path("../data/processed")
processed_dir.mkdir(exist_ok=True)

# Salvar conjuntos de treino e teste
X_train.to_csv(processed_dir / "X_train.csv", index=False)
X_test.to_csv(processed_dir / "X_test.csv", index=False)
y_train.to_csv(processed_dir / "y_train.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)

# Salvar o scaler para uso posterior
import pickle
with open(processed_dir / "scaler.pkl", 'wb') as f:
    pickle.dump(scaler, f)

print("="*80)
print("DADOS PROCESSADOS SALVOS")
print("="*80)

print(f"\n✓ X_train.csv: {X_train.shape}")
print(f"✓ X_test.csv: {X_test.shape}")
print(f"✓ y_train.csv: {y_train.shape}")
print(f"✓ y_test.csv: {y_test.shape}")
print(f"✓ scaler.pkl: StandardScaler object")

print(f"\nLocalização: {processed_dir.absolute()}")

## Seção 8: Resumo do Pré-Processamento

In [ ]:
print("\n" + "="*80)
print("RESUMO DO PRÉ-PROCESSAMENTO")
print("="*80)

print("""
1. TRATAMENTO DE VALORES FALTANTES:
   ✓ Variáveis numéricas: Imputação por MEDIANA
   ✓ Variáveis categóricas: Imputação por MODA
   ✓ Resultado: 0 valores faltantes

2. TRATAMENTO DE OUTLIERS:
   ✓ Decisão: MANTER outliers
   ✓ Motivo: Podem ser casos clínicos genuínos
   ✓ Gradient boosting é robusto a outliers

3. CODIFICAÇÃO DE CATEGÓRICAS:
   ✓ city_hospital: Label Encoding
   ✓ Resultado: Todas as variáveis são numéricas

4. NORMALIZAÇÃO:
   ✓ StandardScaler (Z-score): (x - média) / desvio
   ✓ Resultado: Média ≈ 0, Desvio ≈ 1
   ✓ Melhora convergência do modelo

5. DIVISÃO TREINO/TESTE:
   ✓ 80% treino, 20% teste
   ✓ Estratificado: Mantém proporção de óbitos
   ✓ Random state = 42: Reprodutível

6. DADOS SALVOS:
   ✓ X_train.csv, X_test.csv
   ✓ y_train.csv, y_test.csv
   ✓ scaler.pkl (para usar em novos dados)

PRÓXIMOS PASSOS:
→ Treinar modelos: LightGBM, XGBoost, CatBoost
→ Avaliar desempenho
→ Análise SHAP
→ Análise de justiça algorítmica
""")

print("="*80)
print("Pré-processamento concluído! Prossiga para o notebook 03_modeling_lightgbm.ipynb")
print("="*80)